In [47]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import os
import warnings
from sklearn.metrics.pairwise import cosine_similarity
from sklearn import preprocessing
import math as mt
from scipy.stats import skew

In [48]:
class Network:
    def __init__(self, name, adjacency_matrix):
        self.name = name
        self.adjacency_matrix = adjacency_matrix
        self.graph = self.create_graph()
        self.mean_in_degree = None
        self.mean_out_degree = None
        self.num_isolated_nodes = None
        self.num_nodes = None
        self.num_edges = None
        self.edge_weights_sum_abs = None
        self.edge_weights_sum = None
        self.num_input_nodes = None
        self.num_output_nodes = None
        self.flow_dynamics = None  # To store flow dynamics (shortest path lengths)
        self.min_flow_length = None
        self.max_flow_length = None

    def create_graph(self):
        G = nx.DiGraph()  # Instantiate as a directed graph
        for i in range(len(self.adjacency_matrix)):
            for j in range(len(self.adjacency_matrix[i])):
                if self.adjacency_matrix[i][j] != 0:
                    G.add_edge(i, j, weight=self.adjacency_matrix[i][j])
            if all(val == 0 for val in self.adjacency_matrix[i]):
                G.add_node(i)
        return G

    def calculate_degrees(self):
        in_degrees = [self.graph.in_degree(node) for node in self.graph.nodes()]
        out_degrees = [self.graph.out_degree(node) for node in self.graph.nodes()]
        self.mean_in_degree = np.mean(in_degrees)
        self.mean_out_degree = np.mean(out_degrees)
    
    def calculate_num_nodes(self):
        self.num_nodes = len(self.graph.nodes())

    def calculate_num_edges(self):
        self.num_edges = self.graph.number_of_edges()
        
    def calculate_isolated_nodes(self):
        self.num_isolated_nodes = len(list(nx.isolates(self.graph)))

    def calculate_edge_weights_sum_abs(self):
        self.edge_weights_sum_abs = sum(abs(weight) for _, _, weight in self.graph.edges(data='weight'))

    def calculate_edge_weights_sum(self):
        self.edge_weights_sum = sum(weight for _, _, weight in self.graph.edges(data='weight'))
    
    def count_input_output_nodes(self):
        self.num_input_nodes = sum(1 for node in self.graph.nodes() if self.graph.in_degree(node) == 0)
        self.num_output_nodes = sum(1 for node in self.graph.nodes() if self.graph.out_degree(node) == 0)

    def calculate_flow_dynamics(self):
        # Create a copy of the original graph to avoid modifying the original network
        temp_graph = self.graph.copy()

        # Find the minimum edge weight (including negative weights)
        min_weight = min([d['weight'] for _, _, d in temp_graph.edges(data=True)])

        # Add a constant to all edge weights to make them non-negative
        weight_shift = abs(min_weight) + 1  # Add 1 to avoid zero weights
        for u, v, d in temp_graph.edges(data=True):
            d['weight'] += weight_shift

        # Calculate shortest path lengths
        self.flow_dynamics = dict(nx.all_pairs_dijkstra_path_length(temp_graph, weight='weight'))

        # Correct the shortest path lengths by subtracting the weight shift
        for source, paths in self.flow_dynamics.items():
            for target, length in paths.items():
                if not np.isnan(length):
                    self.flow_dynamics[source][target] -= weight_shift

        # Find max and min flow lengths
        self.find_max_min_flow_dynamics()

    def find_max_min_flow_dynamics(self):
        min_length = float('inf')
        max_length = float('-inf')

        for source, paths in self.flow_dynamics.items():
            for target, length in paths.items():
                if not np.isnan(length):
                    if length < min_length:
                        min_length = length
                    if length > max_length:
                        max_length = length

        self.min_flow_length = min_length if min_length != float('inf') else None
        self.max_flow_length = max_length if max_length != float('-inf') else No


In [49]:
class Step:
    def __init__(self, step_number):
        self.step_number = step_number
        self.networks = []

    def add_network(self, network):
        self.networks.append(network)


In [50]:
def read_multiple_networks(file_path):
    steps = []
    current_step = None

    with open(file_path, 'r') as file:
        for line in file:
            line = line.strip()
            if line.startswith("st."):
                if current_step:
                    steps.append(current_step)
                step_number = int(line.split('.')[1])
                current_step = Step(step_number)
            elif line.startswith("Network"):
                network_name = line
                adjacency_matrix = []
            elif line == '---':
                network_instance = Network(network_name, adjacency_matrix)
                current_step.add_network(network_instance)
            else:
                row = [float(value) for value in line.split()]
                adjacency_matrix.append(row)

    if current_step:
        steps.append(current_step)

    return steps
def calculate_cosine_similarity(vectors, steps):
    for step_idx, step in enumerate(steps):
        print(f"Step {step.step_number}:")
        for network_idx, network in enumerate(step.networks):
            print(f"{len(vectors)}")
    similarities = cosine_similarity(vectors)
    return similarities

# Calculate subtraction sum for each step and save them in a single text file
def save_cosine_similarities(file_path, steps, cosine_similarities):
    with open(file_path, 'w') as file:
        for step_idx, step in enumerate(steps):
            file.write(f"Step {step.step_number}\n")
            for i in range(len(step.networks)):
                for j in range(i+1, len(step.networks)):
                    similarity = cosine_similarities[step_idx][i][j]
                    file.write(f"{step.step_number}\t{similarity}\n")
            file.write('\n')


In [53]:
file_path = './Outputs/st_8000,nn_500,mu_0.900000,ref_env_=0.500000/numrun_0/'
address = file_path + 'History.txt'

In [54]:
steps = read_multiple_networks(address)

In [55]:
for step in steps:
    for network in step.networks:
        network.calculate_degrees()
        network.calculate_isolated_nodes()
        network.calculate_num_nodes()
        network.calculate_num_edges()
        network.calculate_edge_weights_sum_abs()
        network.calculate_edge_weights_sum()
        network.count_input_output_nodes()
        network.calculate_flow_dynamics()
        network.find_max_min_flow_dynamics()


In [56]:
saving = file_path + 'similarity.txt'
with open(saving, 'w') as file:
    for i in range(len(steps)):
        cos_sum = 0
        num_vectors = 0
        for j in range(len(steps[i].networks)):
            #deg = (steps[i].networks[j].mean_in_degree + steps[i].networks[j].mean_out_degree) / 2.0
            #spec1 = [deg, steps[i].networks[j].num_edges, steps[i].networks[j].num_isolated_nodes, steps[i].networks[j].edge_weights_sum]
            #spec1 = [steps[i].networks[j].num_edges, steps[i].networks[j].num_output_nodes,steps[i].networks[j].num_input_nodes, steps[i].networks[j].num_isolated_nodes, steps[i].networks[j].edge_weights_sum, steps[i].networks[j].edge_weights_sum_abs]
            spec1 = [steps[i].networks[j].max_flow_length, steps[i].networks[j].min_flow_length]
            #print(steps[i].networks[j].edge_weights_sum)
            
            for h in range(j, len(steps[i].networks)):    
                #deg2 = (steps[i].networks[h].mean_in_degree + steps[i].networks[h].mean_out_degree) / 2.0
                #spec2 = [deg2, steps[i].networks[h].num_edges, steps[i].networks[h].num_isolated_nodes,steps[i].networks[h].edge_weights_sum]
                spec2 = [steps[i].networks[h].max_flow_length, steps[i].networks[h].min_flow_length]
                
                cos_similarity = cosine_similarity([spec1], [spec2])
                cos_sum += cos_similarity[0][0]
                #print(cos_similarity)
                num_vectors += 1
            
        average_cosine_similarity = round(float(cos_sum / num_vectors),4)
        print (average_cosine_similarity)
        
        file.write(f"{i}\t{average_cosine_similarity}\n")
        file.flush()


0.9943
0.9979
0.9978
0.9968
0.9963
0.9964
0.9965
0.9963
0.9957
0.9944
0.9943
0.9953
0.9932
0.9957
0.9941
0.9934
0.9931
0.9936
0.9941
0.9936
0.992
0.9945
0.9948
0.9936
0.9932
0.9929
0.993
0.9943
0.9926
0.9936
0.9941
0.9929
0.9946
0.9944
0.9937
0.9942
0.9945
0.9947
0.9931
0.994
0.9934
0.9918
0.9942
0.9936
0.9927
0.9929
0.9925
0.9943
0.9941
0.9936
0.9939
0.9937
0.9944
0.9939
0.994
0.9937
0.9943
0.9937
0.9933
0.9923
0.9933
0.9938
0.9935
0.9943
0.9939
0.9922
0.9933
0.9936
0.9926
0.9937
0.9936
0.993
0.9933
0.9936
0.9927
0.9929
0.9935
0.9937


In [ ]:
similarity = np.loadtxt(file_path + 'similarity.txt')
plt.plot(similarity)

In [ ]:
file_path = './Outputs/st_2000,nn_500,mu_0.200000,ref_env_=-0.100000/numrun_0/'
address = file_path + 'History_du.txt'

In [ ]:
steps = read_multiple_networks(address)

In [ ]:
for i in range (len(steps)):
    for j in range(len(steps[i].networks)):
        steps[i].networks[j].calculate_degrees()
        steps[i].networks[j].calculate_isolated_nodes()
        steps[i].networks[j].calculate_num_nodes()
        steps[i].networks[j].calculate_num_edge()

In [21]:
saving = file_path + 'similarity_du.txt'
with open(saving, 'w') as file:
    for i in range(len(steps)):
        cos_sum = 0
        num_vectors = 0
        for j in range(len(steps[i].networks)):
            deg = (steps[i].networks[j].mean_in_degree + steps[i].networks[j].mean_out_degree) / 2.0
            spec1 = [deg, steps[i].networks[j].num_edges, steps[i].networks[j].num_isolated_nodes]
            
            for h in range(j, len(steps[i].networks)):    
                deg2 = (steps[i].networks[h].mean_in_degree + steps[i].networks[h].mean_out_degree) / 2.0
                spec2 = [deg2, steps[i].networks[h].num_edges, steps[i].networks[h].num_isolated_nodes]
                
                cos_similarity = cosine_similarity([spec1], [spec2])
                cos_sum += cos_similarity[0][0]
                #print(cos_similarity)
                num_vectors += 1
            
        average_cosine_similarity = round(float(cos_sum / num_vectors),4)
        print (average_cosine_similarity)
        
        file.write(f"{i}\t{average_cosine_similarity}\n")
        file.flush()


1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0


In [ ]:
similarity = np.loadtxt(file_path + 'similarity_du.txt')
plt.plot(similarity)

In [18]:
steps[0].networks[1].calculate_isolated_nodes()

In [20]:
steps[0].networks[1].mean_out_degree

2.1

In [17]:
x = [2, 3]
y = [2, 0]

In [19]:
q = cosine_similarity([x],[y])

In [20]:
q

array([[0.5547002]])